# FLUX.2 Live Demo with Gallium

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MarsZDF/gallium/blob/main/flux2_demo.ipynb)

**Live image generation with BFL's FLUX.2 API + experiment tracking with Gallium.**

This notebook demonstrates:
- Generating images with FLUX.2 via the BFL API
- Tracking experiments with gallium
- Parameter sweeps (seed, guidance, model)
- Creating comparison grids

**Requirements:**
- BFL API key (get one at https://api.bfl.ml/)

## Understanding FLUX.2 Parameters

Different FLUX.2 models support different parameters:

| Model | seed | width/height | guidance | steps | Notes |
|-------|------|--------------|----------|-------|-------|
| **flux-dev** | ✓ | ✓ | ✓ (1.5-5.0) | ✓ (1-50) | Best for parameter experimentation |
| **flux-pro-1.1** | ✓ | ✓ | ✗ | ✗ | High quality, fixed settings |
| **flux-2-klein-9b** | ✓ | ✓ | ✗ | ✗ | Fast, good quality |

### Key Parameters (flux-dev)

| Parameter | Range | Description |
|-----------|-------|-------------|
| **seed** | Any integer | Random seed for reproducibility. Same seed + same prompt = same image. |
| **guidance** | 1.5 - 5.0 | How closely to follow the prompt. Higher = stricter adherence. |
| **steps** | 1 - 50 | Number of denoising steps. More steps = higher quality but slower. |
| **width/height** | 256 - 1440 | Output dimensions (multiples of 32). |

### Tips for Good Results
- Use **flux-dev** for parameter sweeps (guidance, steps)
- Use **flux-pro-1.1** or **flux-2-klein** for production (faster, optimized settings)
- Start with **seed sweep** to find a good composition

## Installation

In [ ]:
# Install dependencies
# We pin pillow<12.0 to ensure compatibility with the Colab environment (gradio)
!pip install -q "pillow<12.0" --no-cache-dir
!pip install -q "elemental-gallium[grid] @ git+https://github.com/MarsZDF/gallium.git@v0.9.4" --no-cache-dir
!pip install -q requests

import gallium
import importlib.metadata

installed_version = importlib.metadata.version("elemental-gallium")
if gallium.__version__ != installed_version:
    print(f"⚠️  Kernel has gallium {gallium.__version__} loaded, but {installed_version} is installed.")
    print(f"⚠️  Please RESTART THE RUNTIME to use the new version! (Runtime > Restart session)")
    raise RuntimeError("Kernel restart required to load updated library.")
else:
    print(f"✅ Gallium {gallium.__version__} ready.")

## Configuration

Enter your BFL API key below. You can get one at https://api.bfl.ml/

In [ ]:
import os
from getpass import getpass

def get_api_key():
    """Securely retrieve and validate the API key."""
    key = os.environ.get("BFL_API_KEY")
    if not key:
        key = getpass("Enter your BFL API key: ")
    
    if key:
        key = key.strip()
        # Validate standard BFL key format
        if not key.startswith("bfl_"):
            print("\n⚠️  INVALID API KEY FORMAT")
            print("The key must start with 'bfl_'.")
            key = getpass("Please enter your valid BFL API key: ").strip()
            
    return key

# Enter your BFL API Key
# Get one at https://api.bfl.ml/
BFL_API_KEY = get_api_key()

# API endpoint - change this to use different models
# Options:
#   https://api.bfl.ai/v1/flux-2-klein-9b  (klein 9B - fastest, good quality)
#   https://api.bfl.ai/v1/flux-2-pro       (pro - higher quality)
#   https://api.bfl.ai/v1/flux-pro-1.1     (pro 1.1 - latest)
#   https://api.bfl.ai/v1/flux-dev         (dev - experimental)
BFL_API_ENDPOINT = "https://api.bfl.ai/v1/flux-2-klein-9b"

print(f"Using endpoint: {BFL_API_ENDPOINT}")
print(f"API key configured: {'Yes' if BFL_API_KEY and BFL_API_KEY.startswith('bfl_') else 'No'}")

## Setup

In [ ]:
import base64
import time
from io import BytesIO
from pathlib import Path

import requests
from PIL import Image, ImageDraw, ImageFont

import gallium
import gallium.flux as gf

# Create output directory
Path("outputs").mkdir(exist_ok=True)

# Clean up previous runs' DB for a fresh demo
db_path = Path("flux2_experiments.db")
if db_path.exists():
    db_path.unlink()

# Initialize gallium
gallium.init(str(db_path))

print(f"Gallium version: {gallium.__version__}")
print(f"FLUX.2 models available: {gf.MODELS}")
print(f"Default guidance: {gf.GUIDANCE_DEFAULT}")
print(f"Default steps: {gf.STEPS_DEFAULT}")

## BFL API Client & Visualization Helpers

In [ ]:
def generate_image(
    prompt: str,
    seed: int = 42,
    width: int = 1024,
    height: int = 1024,
    guidance: float = 7.5,
    steps: int = 28,
    api_key: str = BFL_API_KEY,
    endpoint: str = BFL_API_ENDPOINT,
) -> tuple[Image.Image, int]:
    """Generate an image using BFL's FLUX.2 API."""
    headers = {
        "accept": "application/json",
        "x-key": api_key,
        "Content-Type": "application/json",
    }
    
    # Map width/height to aspect ratio for BFL API
    # if width == height:
    #     aspect_ratio = "1:1"
    # elif width > height:
    #     aspect_ratio = "16:9" if width / height > 1.5 else "4:3"
    # else:
    #     aspect_ratio = "9:16" if height / width > 1.5 else "3:4"
    
    payload = {
        "prompt": prompt,
        "seed": seed,
        "width": width,
        "height": height,
        "guidance_scale": guidance,
        "num_inference_steps": steps,
    }
    
    start_time = time.time()
    
    response = requests.post(endpoint, headers=headers, json=payload)
    response.raise_for_status()
    result = response.json()
    
    polling_url = result.get("polling_url")
    if not polling_url:
        raise ValueError(f"No polling_url in response: {result}")
    
    while True:
        time.sleep(0.5)
        status_response = requests.get(polling_url, headers={"accept": "application/json", "x-key": api_key})
        status_response.raise_for_status()
        status = status_response.json()
        
        if status.get("status") == "Ready":
            result = status
            break
        elif status.get("status") in ("Error", "Failed"):
            raise RuntimeError(f"Generation failed: {status}")
        
        progress = status.get("progress")
        if progress is not None:
            print(f"  Progress: {progress:.0%}", end="\r")
    
    duration_ms = int((time.time() - start_time) * 1000)
    
    if "result" in result and "sample" in result["result"]:
        image_url = result["result"]["sample"]
        image_response = requests.get(image_url)
        image_response.raise_for_status()
        image = Image.open(BytesIO(image_response.content))
    else:
        raise ValueError(f"Unexpected response format: {result.keys()}")
    
    return image, duration_ms


def generate_and_track(
    prompt: str,
    seed: int = 42,
    width: int = 1024,
    height: int = 1024,
    guidance: float = 7.5,
    steps: int = 28,
    model: str = "flux.2-klein",
    output_dir: str = "outputs",
    **kwargs,
) -> gallium.Experiment:
    """Generate an image and track it with gallium."""
    image, duration_ms = generate_image(
        prompt=prompt, seed=seed, width=width, height=height,
        guidance=guidance, steps=steps, **kwargs,
    )
    
    path = f"{output_dir}/{model.replace('.', '_')}_{seed}_{int(time.time())}.png"
    image.save(path)
    
    exp_id = gallium.log(
        prompt=prompt, seed=seed, path=path, model=model,
        width=image.width, height=image.height, duration_ms=duration_ms,
        params={"guidance": guidance, "steps": steps},
    )
    
    print(f"Generated: {path} ({duration_ms}ms)")
    results = gallium.find(id=exp_id)
    return results[0] if results else None


def display_with_info(exp, size=600):
    """Display an experiment with its metadata overlay."""
    if not exp or not exp.path:
        print("No experiment to display")
        return
    
    img = Image.open(exp.path)
    
    # Resize for display
    ratio = size / max(img.size)
    new_size = (int(img.width * ratio), int(img.height * ratio))
    img = img.resize(new_size, Image.Resampling.LANCZOS)
    
    # Print metadata
    guidance = exp.params.get('guidance', 'N/A') if exp.params else 'N/A'
    print(f"\n{'='*60}")
    print(f"Prompt: {exp.prompt}")
    print(f"Seed: {exp.seed} | Guidance: {guidance} | Size: {exp.width}x{exp.height}")
    print(f"Duration: {exp.duration_ms}ms | Model: {exp.model}")
    print(f"{'='*60}\n")
    
    display(img)


def display_grid_with_title(grid_img, title, experiments=None):
    """Display a grid with a title and optional experiment summary."""
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    
    if experiments:
        # Show the prompt (assuming all have same prompt for sweeps)
        prompts = set(e.prompt for e in experiments)
        if len(prompts) == 1:
            print(f"Prompt: {list(prompts)[0][:80]}..." if len(list(prompts)[0]) > 80 else f"Prompt: {list(prompts)[0]}")
        print(f"Experiments: {len(experiments)}")
    print()
    
    display(grid_img)

print("API client ready!")

## 1. Single Generation

Generate a single image and track it. We'll display it with full metadata.

In [ ]:
# Generate a single image
exp = generate_and_track(
    prompt="A fluffy kitten astronaut floating in space, surrounded by stars and galaxies, cute and whimsical",
    seed=42,
    guidance=7.5,
)

# Display with metadata
display_with_info(exp, size=700)

## 2. Seed Sweep

**What is a seed sweep?** The seed controls the random noise that starts the generation process. Different seeds produce different compositions, even with the same prompt.

**Why do it?** To find the best composition/layout for your prompt before fine-tuning other parameters.

In [ ]:
# Define the prompt
prompt = "Kittens having a tea party in a tiny garden, wearing tiny hats, magical sunlight, storybook illustration"

# Generate seed variations
seeds = [42, 123, 456, 789]
params_list = gf.seed_sweep(prompt, seeds=seeds)

print(f"Generating {len(seeds)} seed variations...")
print(f"Prompt: {prompt}")
print(f"Seeds: {seeds}")
print()

for params in params_list:
    generate_and_track(
        prompt=params["prompt"],
        seed=params["seed"],
        guidance=params["params"]["guidance"],
        steps=params["params"]["steps"],
        model="flux.2-klein",
    )

In [ ]:
# Create a comparison grid with larger images
tea_party_exps = gallium.find(prompt__contains="tea party")
if tea_party_exps:
    # Use larger max_size for better visibility
    grid = gallium.grid(
        tea_party_exps,
        cols=2,
        max_size=400,  # Larger thumbnails
        padding=15,
        background="#ffffff",  # Light background for best visibility
        labels=[f"seed={e.seed}" for e in tea_party_exps],
        label_font_size=18,  # Larger labels
        label_color="#000000",  # Black text
    )
    grid.save("outputs/seed_sweep.png")
    display_grid_with_title(grid, "SEED SWEEP COMPARISON", tea_party_exps)

## 3. Guidance Scale Sweep

**What is guidance scale?** Controls how strictly the model follows your prompt.

| Guidance | Effect |
|----------|--------|
| **1-5** | More creative, may deviate from prompt |
| **5-10** | Balanced - good default range |
| **10-15** | Strict prompt following |
| **15-20** | Very strict, can cause over-saturation |

In [ ]:
# Guidance sweep with the same seed for fair comparison
# Using more extreme values to show the difference clearly
prompt = "A sleepy kitten curled up in a cozy bookshop, warm lighting, surrounded by old books, dreamy atmosphere"
guidance_values = [1.5, 5.0, 10.0, 20.0]  # More extreme range to see effects

if "flux-2-klein" in BFL_API_ENDPOINT:
    print(f"⚠️  Skipping guidance sweep for {BFL_API_ENDPOINT}")
    print("This model uses fixed guidance. To run this sweep, use the 'flux-dev' endpoint.")
else:
    params_list = gf.guidance_sweep(prompt, guidance_values=guidance_values, seed=42)

    print(f"Generating {len(guidance_values)} guidance variations...")
    print(f"Prompt: {prompt}")
    print(f"Guidance values: {guidance_values}")
    print(f"Fixed seed: 42 (for fair comparison)")
    print()

    for params in params_list:
        generate_and_track(
            prompt=params["prompt"],
            seed=params["seed"],
            guidance=params["params"]["guidance"],
            steps=params["params"]["steps"],
            model="flux.2-pro",
        )

In [ ]:
# Create guidance comparison grid
bookshop_exps = gallium.find(prompt__contains="bookshop")

# Filter for experiments that actually have guidance params
bookshop_exps = [e for e in bookshop_exps if e.params and "guidance" in e.params]

if bookshop_exps:
    # Sort by guidance value for consistent display
    bookshop_exps_sorted = sorted(bookshop_exps, key=lambda e: e.params.get('guidance', 0))
    
    grid = gallium.grid(
        bookshop_exps_sorted,
        cols=2,
        max_size=400,
        padding=15,
        background="#ffffff",  # Light background
        labels=[f"guidance={e.params.get('guidance', '?')}" for e in bookshop_exps_sorted],
        label_font_size=18,
        label_color="#000000",  # Black text
    )
    grid.save("outputs/guidance_sweep.png")
    display_grid_with_title(grid, "GUIDANCE SCALE COMPARISON", bookshop_exps_sorted)
else:
    print("No guidance sweep experiments found (skipped or model does not support guidance).")

## 3b. Iterating on the Prompt

Let's change the subject slightly to see how tracking helps us manage different experimental branches. We'll swap "kittens" for "puppies" but keep the tea party theme.

In [ ]:
# Define a variation of the prompt
prompt_variation = "Puppies having a tea party in a tiny garden, wearing tiny hats, magical sunlight, storybook illustration"

# Generate with a known good seed from before
generate_and_track(
    prompt=prompt_variation,
    seed=42,
    guidance=7.5,
)

print("Generated prompt variation. Gallium now tracks both 'Kittens' and 'Puppies' experiments.")

## 4. View All Experiments

See everything you've generated with full metadata.

In [ ]:
# View all experiments with detailed info
all_exps = gallium.find()
print(f"{'='*80}")
print(f"  EXPERIMENT LOG - Total: {len(all_exps)} experiments")
print(f"{'='*80}")
print()

for exp in gallium.recent(10):
    guidance = exp.params.get('guidance', 'N/A') if exp.params else 'N/A'
    starred = '⭐ ' if exp.starred else '   '
    print(f"{starred}[{exp.id:3d}] seed={exp.seed:5d} | guidance={guidance:5} | {exp.duration_ms:5d}ms")
    print(f"        {exp.prompt[:70]}{'...' if len(exp.prompt) > 70 else ''}")
    print()

## 5. Star Your Favorites

Mark the best experiments for easy retrieval later.

In [ ]:
# Automatically star the most recent experiment for demonstration
recent_exps = gallium.recent(1)
if recent_exps:
    best_exp = recent_exps[0]
    print(f"Starring experiment #{best_exp.id}...")
    
    gallium.star(best_exp.id)
    gallium.annotate(best_exp.id, "Best composition from the seed sweep (Auto-starred by demo)")
else:
    print("No experiments found to star.")

# Show starred experiments
starred = gallium.find(starred=True)
print(f"\nTotal starred experiments: {len(starred)}")
for exp in starred:
    print(f"  ⭐ [{exp.id}] {exp.prompt[:50]}...")
    if exp.notes:
        print(f"     Note: {exp.notes}")

# Create a grid of your favorites
if starred:
    fav_grid = gallium.grid(
        starred,
        cols=3,
        max_size=300,
        padding=15,
        background="#ffffff",
        labels=[f"#{e.id} {(e.notes or '')[:25] + '...' if len(e.notes or '') > 25 else (e.notes or '')}" for e in starred],
        label_font_size=16,
        label_color="#000000"
    )
    display_grid_with_title(fav_grid, "YOUR FAVORITES")

## 6. Matrix Grid - Compare Across Two Dimensions

Matrix grids let you compare experiments across two parameters simultaneously (e.g., different prompts × different seeds).

In [ ]:
# Create matrix comparing different prompts vs seeds
# First, let's fill in any missing combinations to make a complete matrix
all_exps = gallium.find()

# Identify unique prompts and seeds from our session
unique_prompts = sorted(list(set(e.prompt for e in all_exps)))
unique_seeds = sorted(list(set(e.seed for e in all_exps if e.seed is not None)))

print(f"Found {len(unique_prompts)} prompts and {len(unique_seeds)} seeds.")
print("Checking for missing combinations to ensure full matrix...")

# Fill the gaps (limit to first 3 prompts/seeds to avoid excessive API usage if lists are long)
target_prompts = unique_prompts[:3]
target_seeds = unique_seeds[:4]

for prompt in target_prompts:
    for seed in target_seeds:
        # Check if we already have this combo
        if not gallium.find(prompt__exact=prompt, seed=seed):
            print(f"Generating missing combo: Prompt='{prompt[:20]}...' Seed={seed}")
            try:
                generate_and_track(prompt=prompt, seed=seed)
            except Exception as e:
                print(f"Failed to generate: {e}")

# Refresh experiments list
all_exps = gallium.find()

if len(all_exps) >= 4:
    print(f"\n{'='*60}")
    print(f"  MATRIX GRID: Prompt × Seed")
    print(f"{'='*60}")
    
    matrix = gallium.matrix_grid(
        all_exps,
        rows="seed",
        cols="prompt",
        max_size=300,  # Larger cells
        show_labels=True,
        label_font_size=14,
        label_color="#000000",  # Black labels
        background="#ffffff",   # White background
        max_label_length=25,    # Longer labels for prompts
    )
    matrix.save("outputs/matrix_comparison.png")
    display(matrix)
else:
    print("Need at least 4 experiments for a matrix grid. Generate more above!")

## 7. Export Results

Export your experiments for sharing or further analysis.

In [ ]:
# Export to different formats
gallium.export("csv", path="outputs/experiments.csv")
gallium.export("json", path="outputs/experiments.json")
gallium.export("html", path="outputs/gallery.html", title="FLUX.2 Experiments")

print("Exported to:")
print("  - outputs/experiments.csv   (spreadsheet)")
print("  - outputs/experiments.json  (programmatic access)")
print("  - outputs/gallery.html      (visual gallery)")

# Preview the CSV data
print(f"\n{'='*60}")
print("  CSV PREVIEW (First 5 lines)")
print(f"{'='*60}")
with open("outputs/experiments.csv", "r") as f:
    for i, line in enumerate(f):
        if i < 5:
            print(line.strip())
        else:
            break

## Summary

### What We Covered

1. **Parameter Understanding** - seed, guidance, steps explained
2. **Single Generation** - Generate and track with full metadata
3. **Seed Sweep** - Find the best composition
4. **Guidance Sweep** - Fine-tune prompt adherence
5. **Experiment Management** - View, star, and annotate
6. **Matrix Comparison** - Compare across two dimensions
7. **Export** - Save to CSV, JSON, or HTML gallery

### Key Parameters Reference

| Parameter | Default | Description |
|-----------|---------|-------------|
| seed | random | Reproducibility control |
| guidance_scale | 7.5 | Prompt adherence (1-20) |
| num_inference_steps | 28 | Quality vs speed (20-50) |
| aspect_ratio | 1:1 | Output dimensions |

### Next Steps

- Try `gf.aspect_ratio_sweep()` for different image dimensions
- Change `BFL_API_ENDPOINT` to try different FLUX.2 models
- See [examples/](https://github.com/MarsZDF/gallium/tree/main/examples) for more integration patterns

## Wrap Up: Recommended Experimental Protocol

Based on the tools demonstrated above, here is a robust workflow for image generation experiments:

1.  **Seed Sweep (Composition):**
    *   Start here. Fix your prompt and model, then generate 4-8 variations using random seeds.
    *   *Goal:* Find a seed that produces a composition/layout you like.

2.  **Guidance Sweep (Adherence):**
    *   Take your best seed and prompt.
    *   Sweep guidance scale (e.g., 2.0, 3.5, 5.0, 7.5) using a controllable model like `flux-dev`.
    *   *Goal:* Balance creativity vs. prompt adherence. Too low = chaotic; Too high = stiff/saturated.

3.  **Refinement (Steps/Sampler):**
    *   (Optional) If using a dev model, sweep step counts (e.g., 20 vs 30 vs 50) to optimize quality vs speed.

4.  **Matrix Comparison:**
    *   Once you have a good setup, cross-reference your best prompts against your best seeds using a Matrix Grid.
    *   *Goal:* Verify that your prompt works reliably across different random noises, not just one lucky seed.

5.  **Tracking & Persistence:**
    *   **Rule of Thumb:** If it's not logged, it didn't happen. Gallium only tracks what you explicitly pass to `gallium.log()`.
    *   You can track images generated by *any* tool (ComfyUI, external scripts, manual downloads) by logging them with `gallium.log(prompt="...", path="/path/to/image.png")`.
    *   Use `gallium.star()` and `gallium.annotate()` immediately to capture your qualitative judgments before you forget.

6.  **Curate & Export:**
    *   Export the full log to CSV/HTML for documentation or sharing with your team.